# 04 - Annexe: Good to Know

**Goal:** Fill in the gaps from Phases 0-3. These are topics that were mentioned but not fully explained, or practical knowledge that strengthens your understanding before applying everything in the real codebase.

Organized by phase. Read the ones relevant to what you're working on — you don't need to memorize all of this.

---

## Phase 0 Gaps: Concepts

---

### Backpropagation: How It Actually Works

In Phase 0, we said "backprop calculates gradients using the chain rule." Here's what that actually means.

Imagine a simple network: `input → multiply by w1 → multiply by w2 → loss`

```
Forward pass (left to right):
x=2 → [×w1=3] → h=6 → [×w2=4] → y=24 → [loss=(y-20)²] → loss=16

Backward pass (right to left, applying chain rule):
d(loss)/dy    = 2(y-20)     = 2(24-20)   = 8
d(loss)/dw2   = d(loss)/dy × h           = 8 × 6    = 48
d(loss)/dw1   = d(loss)/dy × w2 × x      = 8 × 4 × 2 = 64
```

The **chain rule** just means: to get the gradient of an early parameter, multiply all the gradients along the path from loss back to that parameter.

PyTorch does this automatically when you call `.backward()` — it builds a computation graph during the forward pass, then walks it backwards.

In [ ]:
import torch

# Manual backprop example
x = torch.tensor(2.0)
w1 = torch.tensor(3.0, requires_grad=True)
w2 = torch.tensor(4.0, requires_grad=True)

# Forward
h = x * w1        # 6
y = h * w2         # 24
loss = (y - 20)**2 # 16

# Backward (PyTorch computes all gradients)
loss.backward()

print(f"d(loss)/dw2 = {w2.grad:.1f}")  # 48
print(f"d(loss)/dw1 = {w1.grad:.1f}")  # 64

# Verify manually
print(f"\nManual check:")
print(f"d(loss)/dw2 = 2*(24-20) * 6 = {2*(24-20) * 6}")  # 48
print(f"d(loss)/dw1 = 2*(24-20) * 4 * 2 = {2*(24-20) * 4 * 2}")  # 64

---

### Batch Normalization

Used everywhere in modern networks (including your production code) but never explained in the notebooks.

**Problem:** As a network trains, the distribution of inputs to each layer keeps shifting. Layer 5 has to keep re-adapting to whatever Layer 4 outputs. This slows training.

**Solution:** Normalize each layer's inputs to mean=0, std=1 within each batch.

```
Input batch: [12.5, 8.3, 15.1, 9.7]
                    ↓ normalize
Normalized:  [0.52, -1.22, 1.60, -0.90]   ← mean=0, std=1
                    ↓ scale & shift (learned)
Output:      [γ × 0.52 + β, ...]           ← model can undo the normalization if needed
```

**Key detail:** During training, uses batch statistics. During inference (`model.eval()`), uses running averages computed during training. This is why you **must** call `model.eval()` before inference — otherwise BatchNorm uses the wrong statistics.

```python
# In your production code
model.eval()  # ← switches BatchNorm to use running statistics
with torch.no_grad():
    output = model(input)
```

---

### Learning Rate Schedules

All our notebooks used a fixed learning rate. In practice, you almost always **decay** the learning rate during training.

**Why?** Early in training, big steps help explore. Later, big steps overshoot the optimum.

```
Fixed LR:          ████████████████████  (same speed throughout)
Step decay:        ████████░░░░░░░░    (drop by 10x every N epochs)
Cosine annealing:  ████████▓▓▓▓░░░░   (smooth curve from high to low)
Warmup + cosine:   ░▓████▓▓▓▓░░░░     (start slow, ramp up, then decay)
```

Your production training config uses cosine annealing:
```yaml
# spine_semantic_new_v8.yaml
lr: 1.0e-04  # initial learning rate
```

Common PyTorch schedulers:

```python
# Step: drop LR by 0.1 every 30 epochs
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=30, gamma=0.1)

# Cosine: smooth decay to near-zero
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=100)

# Usage in training loop
for epoch in range(100):
    train_one_epoch()
    scheduler.step()  # ← update LR after each epoch
```

---

## Phase 1 Gaps: PyTorch

---

### Data Augmentation: Why and How

We used transforms in Phase 1 but didn't explain **why** they help.

**Problem:** Small dataset → model memorizes training images → fails on new images.

**Solution:** Randomly transform training images so the model sees variations:

```
Original image:    [spine CT]
                        ↓ random augmentation
Epoch 1 sees:      [rotated 5°, brightness +10%]
Epoch 2 sees:      [flipped, contrast -5%]
Epoch 3 sees:      [scaled 95%, shifted 3px]
```

The model never sees the exact same image twice → forced to learn general features.

**Medical imaging gotcha:** Not all augmentations are valid!

| Augmentation | Natural images | Medical (spine CT) |
|-------------|---------------|-------------------|
| Horizontal flip | Yes | Yes (left-right symmetry) |
| Vertical flip | Yes | **No** (spine doesn't flip upside down) |
| Small rotation (±15°) | Yes | Yes |
| Large rotation (90°) | Sometimes | **No** (anatomy has orientation) |
| Color jitter | Yes | **No** (CT values are physical measurements) |
| Intensity shift | Sometimes | Yes (scanner variation) |
| Elastic deformation | Sometimes | Yes (soft tissue variation) |

Your production code uses MONAI augmentations:
```python
# From training transforms
RandRotated(prob=0.3, range_x=0.26)     # ±15° rotation
RandScaleIntensityd(prob=0.5, factors=0.3)  # intensity variation
RandShiftIntensityd(prob=0.5, offsets=0.1)  # intensity shift
RandCropByPosNegLabeld(...)              # crop around foreground
```

---

### Overfitting: Detection and Prevention

Phase 1 showed train vs val loss curves but didn't explain the full picture.

**How to detect overfitting:**
```
Good training:           Overfitting:

Loss ↑                   Loss ↑
  |\                       |\      val loss goes UP
  | \  train               | \___/‾‾‾‾‾   ← divergence
  |  \_____ val            |  \________
  |   \_____ train         |    train keeps going down
  +----------→ Epoch       +----------→ Epoch
  Both decrease together   Gap = overfitting
```

**Prevention toolkit:**

| Technique | How it works | When to use |
|-----------|-------------|------------|
| Data augmentation | More training variety | Always |
| Early stopping | Stop when val loss stops improving | Always |
| Dropout | Randomly zero neurons during training | When model is too large |
| Weight decay (L2) | Penalize large weights | Always (small value like 1e-5) |
| Smaller model | Fewer parameters to memorize with | When data is very small |
| More data | More examples to learn from | When possible |

**Early stopping in practice:**
```python
best_val_loss = float('inf')
patience = 10  # stop after 10 epochs without improvement
counter = 0

for epoch in range(max_epochs):
    train_loss = train()
    val_loss = validate()
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        counter = 0
        torch.save(model.state_dict(), 'best_model.pth')  # save best
    else:
        counter += 1
        if counter >= patience:
            print(f"Early stopping at epoch {epoch}")
            break
```

---

### Debugging NaN/Inf in Training

You'll eventually see `loss = nan` and wonder what happened.

**Common causes:**

| Cause | Symptom | Fix |
|-------|---------|-----|
| Learning rate too high | Loss explodes to `inf` then `nan` | Reduce LR by 10x |
| Division by zero | Sudden `nan` in one batch | Add epsilon: `x / (y + 1e-8)` |
| Log of zero/negative | `nan` after softmax/log | Use `log_softmax` instead of `log(softmax)` |
| Exploding gradients | Gradients grow each step | Gradient clipping |
| Bad data | `nan` in input data | Check data loading pipeline |

**Gradient clipping** (used in your production training):
```python
# Limit gradient magnitude to prevent explosion
loss.backward()
torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
optimizer.step()
```

**Debug checklist:**
1. Check if input data has NaN: `torch.isnan(input).any()`
2. Check loss value: `if torch.isnan(loss): print("NaN loss!")`
3. Check gradients: `for p in model.parameters(): print(p.grad.max())`
4. Reduce learning rate
5. Try without mixed precision (if using it)

---

## Phase 2 Gaps: Segmentation

---

### IoU (Jaccard) vs Dice

Phase 2 used Dice score. IoU is the other standard metric. They're related but not identical.

```
Prediction:  ████████░░░░
Ground truth:    ████████░░░░
                 ↑↑↑↑↑↑
              Intersection = 6

IoU  = Intersection / Union        = 6 / 14 = 0.43
Dice = 2 × Intersection / (A + B)  = 12 / 20 = 0.60
```

Dice is always ≥ IoU for the same prediction. They rank models the same way (if model A has higher Dice than B, it also has higher IoU).

**When to use which:**
- **Dice:** Standard in medical imaging segmentation (your production code)
- **IoU:** Standard in computer vision (PASCAL VOC, COCO challenges)
- **Both:** Report both in papers/FDA submissions for completeness

**Conversion:** `IoU = Dice / (2 - Dice)`

---

### Class Imbalance: Beyond Dice Loss

Dice loss handles imbalance well, but it's not the only option.

**The problem in numbers:**
```
Spine CT volume (512 × 512 × 300 voxels):
  Background:     ~97% of voxels     (75M voxels)
  All vertebrae:   ~3% of voxels     (2.3M voxels)
  Single vertebra: ~0.1% of voxels   (78K voxels)
```

**Toolkit for class imbalance:**

| Strategy | How it works | MONAI equivalent |
|----------|-------------|------------------|
| Dice loss | Measures overlap, ignores easy background | `DiceLoss` |
| Weighted CE | Higher penalty for rare classes | `CrossEntropyLoss(weight=...)` |
| Focal loss | Down-weights easy samples, focuses on hard ones | `FocalLoss` |
| DiceCE | Best of both (your production choice) | `DiceCELoss` |
| Foreground sampling | Crop patches centered on structures | `RandCropByPosNegLabeld` |

Your production config:
```yaml
foreground_ratio: 2  # 2:1 ratio of foreground to background patches
loss: DiceCELoss     # combined loss
```

This combination (DiceCE + foreground sampling) is the standard approach in medical segmentation.

---

### 2D vs 3D Segmentation

Phase 2 trained a 2D U-Net. Production uses 3D. Here's why and when.

```
2D: Process each slice independently
┌─────┐ ┌─────┐ ┌─────┐
│Slice│ │Slice│ │Slice│   Each slice → model → prediction
│  1  │ │  2  │ │  3  │   No context between slices
└─────┘ └─────┘ └─────┘

3D: Process volume as a block
┌─────────────────┐
│  Volume patch    │
│  128×128×96      │       Full 3D context
│  (x, y, z)      │       Sees across slices
└─────────────────┘
```

| | 2D | 3D |
|--|-----|-----|
| Memory | Low (~1 GB) | High (~4-8 GB per patch) |
| Speed | Fast | Slow (3D convolutions) |
| Cross-slice context | None | Full |
| Best for | Pathology, X-ray, photos | CT, MRI (volumetric) |
| Spine segmentation | Bad (vertebra spans many slices) | **Required** |

**Why 3D is required for spine:** A vertebra spans 10-30 slices in a CT. A 2D model looking at one slice can't tell if it's seeing the top of L4 or the bottom of L3 — they look identical. The 3D model sees the full vertebral body and its neighbors.

**2.5D (middle ground):** Process 3-5 adjacent slices together. Gives some cross-slice context with 2D-like memory usage. Not used in your production code but common in research.

---

### Transfer Learning in Medical Imaging

Phase 2 MONAI notebook mentioned pre-trained weights. Here's the full picture.

**The idea:** Instead of training from random weights, start from weights trained on a large dataset.

```
From scratch:     Random weights → Train 500 epochs → Good model
Transfer learning: Pre-trained weights → Train 100 epochs → Better model
```

**But does ImageNet help for CT scans?**

Surprisingly, **yes, a little.** Even though ImageNet is cats and dogs, the early layers learn basic features (edges, textures) that transfer. But medical-specific pre-training is much better:

| Pre-training source | Usefulness for spine CT |
|---------------------|------------------------|
| Random initialization | Baseline |
| ImageNet (natural images) | Slight improvement (early features transfer) |
| RadImageNet (medical images) | Good improvement |
| Large CT dataset (e.g., TotalSegmentator) | Best improvement |

**For your production code:** SwinUNETR has pre-trained weights available from MONAI Model Zoo, trained on large CT datasets. Using them as initialization would likely improve training speed and final accuracy.

```python
from monai.networks.nets import SwinUNETR

model = SwinUNETR(
    img_size=(128, 128, 96),
    in_channels=1,
    out_channels=5,
)

# Load pre-trained weights (encoder only)
weight = torch.load("swinunetr_pretrained.pt")
model.load_state_dict(weight, strict=False)  # strict=False: OK if decoder shapes differ
```

---

### Evaluation Metrics Beyond Dice

Dice tells you "how much overlap" but not "how far off the boundaries are." For clinical use, you need more.

```
Case A (Dice = 0.90):               Case B (Dice = 0.90):
┌──────────────────┐                 ┌──────────────────┐
│  ▓▓▓▓▓▓▓▓▓▓░░    │                 │  ▓▓▓▓▓▓▓▓▓▓      │
│  ▓▓▓▓▓▓▓▓▓▓░░    │                 │  ▓▓▓▓▓▓▓▓▓▓      │
│  ▓▓▓▓▓▓▓▓▓▓      │                 │  ▓▓▓▓▓▓▓▓▓▓      │
│  ▓▓▓▓▓▓▓▓▓▓      │                 │  ▓▓▓▓▓▓▓▓▓▓░░░░  │
└──────────────────┘                 └──────────────────┘
  Boundary error: 2px everywhere      Boundary error: 0 except 4px spike
  Hausdorff distance: 2               Hausdorff distance: 4
```

Same Dice, but Case B has a dangerous outlier error. A surgeon trusting Case B could cut in the wrong place.

**Key metrics for medical segmentation:**

| Metric | What it measures | Why it matters |
|--------|-----------------|----------------|
| **Dice** | Overall overlap | General quality |
| **IoU** | Overall overlap (stricter) | Standard in CV |
| **Hausdorff Distance (HD95)** | Worst-case boundary error (95th percentile) | Catches dangerous outliers |
| **Average Surface Distance** | Mean boundary error | Average precision |
| **Per-class Dice** | Dice for each vertebra | Which structures are hard? |
| **Sensitivity** | What % of true structures were found | Missing a vertebra |
| **Specificity** | What % of negatives are correct | False alarms |

**For FDA submissions**, you'll want to report Dice + HD95 + per-class breakdown at minimum.

```python
# MONAI provides all of these
from monai.metrics import DiceMetric, HausdorffDistanceMetric, SurfaceDistanceMetric

dice_metric = DiceMetric(include_background=False, reduction="mean_batch")
hd_metric = HausdorffDistanceMetric(include_background=False, percentile=95)
```

---

## Phase 3 Gaps: Transformers

---

### Positional Encoding: Why It Exists

Phase 3 used learnable position embeddings without explaining the alternatives.

**The problem:** Attention treats input as a **set** (unordered). Without position information, the model can't tell if a patch is at position 1 or position 100.

```
Without position encoding:
  Patch at position 1:  [0.3, 0.7, 0.1]  ← model sees this
  Patch at position 50: [0.3, 0.7, 0.1]  ← same! Can't distinguish

With position encoding:
  Patch at position 1:  [0.3, 0.7, 0.1] + [0.0, 0.0, 0.0]  ← unique
  Patch at position 50: [0.3, 0.7, 0.1] + [0.9, 0.4, 0.7]  ← unique
```

**Three approaches:**

| Type | How | Used in |
|------|-----|--------|
| Sinusoidal (fixed) | Math formula: `sin(pos / 10000^(2i/d))` | Original Transformer |
| Learnable (absolute) | Random init, learned during training | ViT |
| Relative position bias | Learn offsets between positions, not absolute | **Swin Transformer** |

Your production model (Swin) uses **relative position bias** — it learns that "the patch 2 positions to the right" has a certain relationship, regardless of absolute position. This is better for medical imaging because CT scans can be different sizes.

---

### Attention Maps: What They Show You

Phase 3 computed attention weights but didn't discuss how to interpret them.

**What you can do:** Visualize which parts of the input the model focuses on for each output.

```
Input CT slice:           Attention map:
┌──────────────────┐      ┌──────────────────┐
│                  │      │                  │
│    ╔══╗          │      │    ████          │
│    ║L3║          │      │    ████          │
│    ╚══╝          │      │    ████          │
│    ╔══╗          │      │    ▓▓▓▓          │
│    ║L4║          │      │    ▓▓▓▓          │
│    ╚══╝          │      │                  │
│                  │      │                  │
└──────────────────┘      └──────────────────┘
                          Dark = high attention
                          Model focuses on vertebrae and neighbors
```

**Useful for:**
- **Debugging:** Is the model looking at the right anatomy?
- **Trust:** Showing clinicians where the model is looking
- **FDA:** Evidence that the model uses relevant features

**Caveat:** Attention ≠ importance. High attention on a region doesn't necessarily mean that region caused the prediction. For proper explanations, combine with gradient-based methods (saliency maps, GradCAM).

---

## Cross-Phase Gaps

---

### Mixed Precision Training

Mentioned in Phase 0 (GPU notebook) and used in production but never fully explained.

**The idea:** Use float16 (half precision) for most computation, float32 (full precision) only where needed.

```
float32: 32 bits per number → more precise, more memory
float16: 16 bits per number → less precise, half the memory

Mixed: forward pass in float16 (fast), loss/gradients in float32 (stable)
```

**Why it matters for 3D medical imaging:**
- A 128×128×96 patch with SwinUNETR uses ~4 GB in float32
- Same patch in mixed precision: ~2.5 GB
- Means you can use larger batches or higher resolution

**How it works in PyTorch:**
```python
scaler = torch.amp.GradScaler()  # handles loss scaling

for batch in dataloader:
    optimizer.zero_grad()
    
    # Forward pass in float16 (automatic)
    with torch.amp.autocast('cuda'):
        output = model(batch['image'])
        loss = loss_fn(output, batch['label'])
    
    # Backward pass with loss scaling (prevents underflow)
    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()
```

**Loss scaling** is needed because float16 has a small range — tiny gradients can underflow to zero. The scaler multiplies the loss by a large number before backward, then divides the gradients back before the optimizer step.

Your production training likely uses this (check for `amp` in `training_semantic.py`).

---

### Inference Optimization: Making Models Faster

Training is slow and that's fine. Inference needs to be fast — a radiologist won't wait 5 minutes.

**Optimization hierarchy (easiest to hardest):**

```
1. torch.no_grad()           ← 30% faster (skip gradient tracking)
   ↓
2. model.eval()              ← correct BatchNorm/Dropout behavior
   ↓
3. Mixed precision inference ← 1.5-2x faster on modern GPUs
   ↓
4. ONNX Runtime              ← 2-4x faster than native PyTorch
   ↓
5. TensorRT (NVIDIA)         ← 3-5x faster (GPU optimized)
   ↓
6. Quantization (INT8)       ← 2-4x smaller model, faster on CPU
   ↓
7. Pruning                   ← remove unimportant weights
   ↓
8. Knowledge distillation    ← train smaller model to mimic large one
```

Your production code already does 1 and 2. ONNX export (covered in notebook 03) would be the next big win.

**Quantization example:**
```python
# Post-training quantization (simplest)
quantized_model = torch.quantization.quantize_dynamic(
    model,
    {torch.nn.Linear},  # which layers to quantize
    dtype=torch.qint8
)
# Model is now ~4x smaller and faster on CPU
```

**Current inference times (from production):**
```
Region model (2mm):   ~10-20s
Instance model (1mm): ~20-60s
SegRefine:            ~5-10s
Total:                ~35-90s
```

With ONNX + TensorRT, these could potentially drop by 50-70%.

---

### Multi-Scale / Cascade: Why Two Models?

Your production pipeline uses two models in sequence. This pattern is common but wasn't explained in the learning notebooks.

**Why not one model?**

```
Option A: Single model at 1mm
  Input: 512 × 512 × 300 = 78M voxels
  GPU memory needed: ~32 GB (won't fit on most GPUs)
  Output: 29 vertebrae labels
  Problem: too much memory, too much to learn at once

Option B: Two models (your production approach)
  Model 1 (region, 2mm):    256 × 256 × 150 = 10M voxels → 5 regions
  Model 2 (instance, 1mm):  128 × 128 × 96 patches → specific vertebrae
  GPU memory: ~4-8 GB each (fits easily)
```

**The cascade pattern:**
```
Full CT scan (512×512×300 at 1mm)
        │
        ▼ resample to 2mm
┌─────────────────┐
│  Region Model   │  "Where is the thoracic region?"
│  (coarse, fast)  │  Output: bounding box per region
└───────┬─────────┘
        │
        ▼ crop + resample to 1mm
┌─────────────────┐
│ Instance Model  │  "Which specific vertebra is this?"
│ (fine, focused)  │  Output: per-vertebra labels
└───────┬─────────┘
        │
        ▼ merge all regions
    Final output
```

**Benefits:**
- Each model has a simpler job (fewer classes, smaller input)
- Can train each independently (easier debugging)
- Can upgrade one without touching the other
- Fits in reasonable GPU memory

**Trade-off:** Errors cascade. If the region model misidentifies a region, the instance model gets wrong input. Your production mitigates this with overlap between regions.

---

## Quick Reference Card

Things you'll want to look up again when working on the real codebase:

```
TRAINING CHECKLIST
──────────────────
□ model.train() before training, model.eval() before validation
□ torch.no_grad() during validation and inference
□ Set random seed for reproducibility
□ Use LR scheduler (cosine annealing or step decay)
□ Gradient clipping for stability
□ Monitor train AND val loss — watch for divergence
□ Save best model (by val metric), not last
□ Patient-level splits, never scan-level

DEBUGGING CHECKLIST
────────────────────
□ NaN loss? → Check LR, check data, check loss function
□ Loss not decreasing? → LR too low, or model too small
□ Val loss going up? → Overfitting. Add augmentation/early stopping
□ GPU out of memory? → Reduce batch size, use mixed precision
□ Different results each run? → Set all seeds, deterministic mode

MEDICAL IMAGING SPECIFICS
──────────────────────────
□ Use DiceCE loss (not plain CE)
□ Use foreground-weighted sampling
□ Report Dice + HD95 + per-class metrics
□ Validate on different scanners/hospitals if possible
□ Augmentations must preserve anatomy
□ 3D for volumetric data (CT/MRI)
```

---

*This annexe complements the main learning notebooks. Refer back to the specific phase notebooks for deeper explanations and hands-on code.*